# The EYE/N3 certificate checker, in a notebook

`eye/check.py` is a plain Python module, so this notebook **imports it** and
calls its functions directly — the same code the command line runs, without a
subprocess per check. The one place we still shell out is the CLI itself, to
show the exit-code contract a pipeline (and the *Try It* page) actually sees.

What it shows:

1. an honest certificate is **accepted**;
2. a forged one is **refused**, naming the first bad step;
3. the two certificate *kinds* (`oo-cert` OWL-RL vs `oo-horn`);
4. the optional **proven anchor** (Rocq), and a real false pass it catches.

**What an `accept` means here.** `check.py` is the untrusted driver; the trusted
part is `checker.n3`, run by EYE. It carries **no machine-checked theorem**
(`OOCert.certificate_sound` is about the Lean function and is not reproved here).
An accept means *the rules licensed every step*, not *it is entailed*. With the
Rocq anchor set — and only for `oo-horn` — the accept gains the theorem.

## 0 · Import the checker

In [1]:
import importlib.util, json, subprocess, sys, re
from pathlib import Path

# the notebook lives in eye/examples/; the checker is one level up
EYE = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
CHECK = EYE / "check.py"
O = EYE.parent                      # the open-ontologies checkout
FIX = O / "tests" / "fixtures"
ROCQ = O / "rocq" / "build" / "oo-horn-rocq"
assert CHECK.exists(), f"no check.py at {CHECK}"

# Import check.py by path (it is not a package). This is the code the CLI runs.
spec = importlib.util.spec_from_file_location("eyecheck", CHECK)
eye = importlib.util.module_from_spec(spec)
spec.loader.exec_module(eye)
print("imported :", eye.__name__, "from", CHECK)
print("fixtures :", FIX)
print("anchor   :", ROCQ, "(built)" if ROCQ.exists() else "(not built)")

imported : eyecheck from /Users/jim/agent-zero/agent-zero-1/usr/projects/pup/open-ontologies/eye/check.py
fixtures : /Users/jim/agent-zero/agent-zero-1/usr/projects/pup/open-ontologies/tests/fixtures
anchor   : /Users/jim/agent-zero/agent-zero-1/usr/projects/pup/open-ontologies/rocq/build/oo-horn-rocq (built)


## 1 · Check a certificate, in process

`check.py` exposes exactly the pieces `main()` uses:

- `parse_asserted(text)` -> list of triples;
- `load_derivations(path)` -> `(format, steps, refute_line)`, auto-detecting
  `oo-cert` / `oo-horn` / `oo-refute` from the file itself;
- `check_steps(asserted, steps)` -> `(ok, first_bad_index)` — the EYE run.

The helper below calls those and returns `(format, ok, bad_index, steps)`. It
decides nothing itself.

In [2]:
def check(asserted_path, cert_path):
    asserted = eye.parse_asserted(Path(asserted_path).read_text())
    fmt, steps, _ = eye.load_derivations(Path(cert_path))
    ok, idx = eye.check_steps(asserted, steps)
    return fmt, ok, idx, steps

def show(label, asserted_path, cert_path):
    fmt, ok, idx, steps = check(asserted_path, cert_path)
    verdict = "ACCEPT" if ok else f"REJECT at step {idx}"
    print(f"{label:<44} {fmt:<8} {verdict}  ({len(steps)} step(s))")
    if not ok:
        rule, concl, prem = steps[idx]
        print("     rule:", rule)
        print("     conclusion:", " ".join(concl))
    return fmt, ok, idx, steps

### 1a · An honest certificate is accepted

In [3]:
show("honest rdfs9 certificate", FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv")

honest rdfs9 certificate                     oo-cert  ACCEPT  (1 step(s))


('oo-cert',
 True,
 None,
 [('rdfs9',
   ('<http://ex.org/leo>',
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>',
    '<http://ex.org/Carnivore>'),
   [('<http://ex.org/leo>',
     '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>',
     '<http://ex.org/Lion>'),
    ('<http://ex.org/Lion>',
     '<http://www.w3.org/2000/01/rdf-schema#subClassOf>',
     '<http://ex.org/Carnivore>')])])

### 1b · A forged one is refused

Same certificate, change **only the conclusion object** (`Carnivore` ->
`Herbivore`), both premises exactly as they were. The checker names the first
bad step. This is the red edge the *Try It* page draws.

In [4]:
src = (FIX / "refute/derivations.tsv").read_text()
forged = re.sub("Carnivore", "Herbivore", src, count=1)   # first occurrence only
forged_path = EYE / "examples" / "forged_derivations.tsv"
forged_path.write_text(forged)
print("before:", src.splitlines()[0].split("\t")[:4])
print("after :", forged.splitlines()[0].split("\t")[:4])
print()
show("forged certificate", FIX / "refute/asserted.tsv", forged_path)

before: ['rdfs9', '<http://ex.org/leo>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://ex.org/Carnivore>']
after : ['rdfs9', '<http://ex.org/leo>', '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>', '<http://ex.org/Herbivore>']

forged certificate                           oo-cert  REJECT at step 0  (1 step(s))
     rule: rdfs9
     conclusion: <http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>


('oo-cert',
 False,
 0,
 [('rdfs9',
   ('<http://ex.org/leo>',
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>',
    '<http://ex.org/Herbivore>'),
   [('<http://ex.org/leo>',
     '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>',
     '<http://ex.org/Lion>'),
    ('<http://ex.org/Lion>',
     '<http://www.w3.org/2000/01/rdf-schema#subClassOf>',
     '<http://ex.org/Carnivore>')])])

## 2 · Two certificate kinds (auto-detected)

- **`oo-cert`** — `derivations.tsv`: the OWL-RL rule set (29 arms), what the
  engine's `reason --certificate` writes and the *Try It* page checks.
- **`oo-horn`** — `horn.tsv`: the Horn rule set (the Rocq-proven one).

`load_derivations` picks the parser from the file's own shape.

In [5]:
show("oo-horn (Horn rule set)", FIX / "horn/asserted.tsv", FIX / "horn/good.tsv")

oo-horn (Horn rule set)                      oo-horn  ACCEPT  (1 step(s))


('oo-horn',
 True,
 None,
 [('rdfs9',
   ('<http://ex.org/a>',
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>',
    '<http://ex.org/B>'),
   [('<http://ex.org/a>',
     '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>',
     '<http://ex.org/A>'),
    ('<http://ex.org/A>',
     '<http://www.w3.org/2000/01/rdf-schema#subClassOf>',
     '<http://ex.org/B>')])])

## 3 · Shape and order are enforced

Rejected from the Lean contract: a premise asserted nowhere and concluded
nowhere; premises in the **wrong order**; `scm-avf2` with the conclusion in the
*natural* direction (the one rule whose conclusion is deliberately reversed —
the natural way round would be unsound); `cls-int1` when one list member's type
is missing. The suite proves all of it; run it with the same CLI a pipeline uses:

In [6]:
p = subprocess.run(["bash", str(EYE / "run_tests.sh")], capture_output=True, text=True, cwd=str(EYE))
print("\n".join(p.stdout.splitlines()[-8:]))
print("suite exit:", p.returncode)

       {"ok": true, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 0, "verdict": {"ok": true, "kernel": "rocq", "verdict": "entailed", "rules": 27, "asserted": 2, "derivations": 1, "theorem": "OOCertRocq.entails_of_builtin_horn", "means": "every conclusion is true in every model of the asserted graph"}}}
PASS [anchor: oo-horn bad_conclusion (caught false pass)] exit=3
       {"ok": false, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 1, "verdict": {"ok": false, "kernel": "rocq", "rules": 27, "asserted": 2, "derivations": 1}}, "anchor_disagrees": true, "error": "the proven oo-horn anchor refused a certificate EYE accepted (possible false pass)"}
PASS [anchor: oo-cert declines, EYE verdict stands] exit=0
       {"ok": true, "asserted": 4, "derivations": 1, "anchor": {"applied": false, "reason": "oo-horn (Horn) checker only; oo-cert uses the OWL-RL rule set (incl. the RDF-list rules it does not cover)"}}
PASS [anchor: broken p

## 4 · The optional proven anchor (Rocq)

`check.py` runs the **Rocq-extracted, machine-checked** `oo-horn` checker
(theorem `entails_of_builtin_horn`) over the same inputs, when `ANCHOR` and
`RULES_TSV` are set. `run_anchor(cert, asserted)` returns its verdict;
`anchor_declined(fmt)` is what the driver reports for the certificate kinds the
anchor does not cover.

We set the module's `ANCHOR`/`RULES_TSV` directly (the CLI sets them from
`OO_HORN_ROCQ`/`OO_RULES`).

In [7]:
def anchor(asserted_path, cert_path, binary=ROCQ, rules=None):
    eye.ANCHOR = str(binary)
    eye.RULES_TSV = str(rules if rules is not None else FIX / "horn/builtin_rules.tsv")
    try:
        return eye.run_anchor(str(cert_path), str(asserted_path))
    finally:
        eye.ANCHOR = None
        eye.RULES_TSV = None

if ROCQ.exists():
    good = anchor(FIX / "horn/asserted.tsv", FIX / "horn/good.tsv")
    print("oo-horn good   -> anchor exit", good["exit"], "|", good["verdict"].get("theorem"))

    bad = anchor(FIX / "horn/asserted.tsv", FIX / "horn/bad_conclusion.tsv")
    print("oo-horn bad    -> anchor exit", bad["exit"], "| EYE says ACCEPT, anchor says REJECT",
          "=> a CAUGHT FALSE PASS" if bad["exit"] != 0 else "")

    print("oo-cert        ->", json.dumps(eye.anchor_declined("oo-cert")))
else:
    print("The Rocq anchor is not built here. Build it with:")
    print("  (cd", O / "rocq", "&& bash build.sh)")

oo-horn good   -> anchor exit 0 | OOCertRocq.entails_of_builtin_horn
oo-horn bad    -> anchor exit 1 | EYE says ACCEPT, anchor says REJECT => a CAUGHT FALSE PASS
oo-cert        -> {"applied": false, "reason": "oo-horn (Horn) checker only; oo-cert uses the OWL-RL rule set (incl. the RDF-list rules it does not cover)"}


### Why `bad_conclusion` matters

That fixture has a binding `b = Z` contradicting its own conclusion object `B`.
EYE **accepts** it (the Horn bindings are dropped when the rule index is mapped
through `asHorn`); the proven checker **refuses** it. That is a textbook **false
pass** — the dangerous direction — surfaced only by running the two together.
The CLI turns this into exit **3** (section 5), so a pipeline cannot read it as
success. `run_tests.sh` keeps it under test.

## 5 · The CLI contract (what the page and pipelines use)

Everything above was in-process. A pipeline sees the exit codes
`main()` returns — `0` accept, `1` reject (JSON names it), `2` parse error, and
`3` **only with the anchor**, when EYE accepts but the theorem refuses. One
subprocess call shows it, and that the anchor is wired through the environment:

In [8]:
import os
E = dict(os.environ)
E["OO_HORN_ROCQ"] = str(ROCQ)
E["OO_RULES"] = str(FIX / "horn/builtin_rules.tsv")

def cli(label, asserted_path, cert_path, env=E):
    p = subprocess.run([sys.executable, str(CHECK), str(asserted_path), str(cert_path)],
                       capture_output=True, text=True, cwd=str(EYE), env=env)
    line = next((l for l in p.stdout.splitlines() if l.startswith("{")), "")
    print(f"{label:<44} exit={p.returncode}")
    print("   ", line[:300] if line else p.stderr[-200:])

cli("CLI: honest oo-cert", FIX / "refute/asserted.tsv", FIX / "refute/derivations.tsv")
cli("CLI: forged oo-cert", FIX / "refute/asserted.tsv", forged_path)
cli("CLI: oo-horn good + anchor", FIX / "horn/asserted.tsv", FIX / "horn/good.tsv")
cli("CLI: oo-horn bad_conclusion + anchor", FIX / "horn/asserted.tsv", FIX / "horn/bad_conclusion.tsv")

CLI: honest oo-cert                          exit=0
    {"ok": true, "asserted": 4, "derivations": 1, "anchor": {"applied": false, "reason": "oo-horn (Horn) checker only; oo-cert uses the OWL-RL rule set (incl. the RDF-list rules it does not cover)"}}
CLI: forged oo-cert                          exit=1
    {"ok": false, "rejected_step": 0, "rule": "rdfs9", "conclusion": "<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Herbivore>", "premises": ["<http://ex.org/leo> <http://www.w3.org/1999/02/22-rdf-syntax-ns#type> <http://ex.org/Lion>", "<http://ex.org/Lion> <http:/
CLI: oo-horn good + anchor                   exit=0
    {"ok": true, "asserted": 2, "derivations": 1, "anchor": {"binary": "oo-horn-rocq", "exit": 0, "verdict": {"ok": true, "kernel": "rocq", "verdict": "entailed", "rules": 27, "asserted": 2, "derivations": 1, "theorem": "OOCertRocq.entails_of_builtin_horn", "means": "every conclusion is true in every mo
CLI: oo-horn bad_conclusion + anch

## 6 · What this does *not* claim

- The `oo-cert` soundness theorem (`OOCert.certificate_sound`) is about the
  **Lean** function and is **not** reproved here. For `oo-cert`, an accept is
  *the rules licensed every step*.
- The Rocq anchor proves a theorem about the **Horn** rule set only
  (`entails_of_builtin_horn`); it applies to `oo-horn`, not `oo-cert`.
- Even with the anchor, the verdict is **conditional on the asserted graph**:
  neither EYE nor Rocq checks the graph is the one the engine actually reasoned
  over.

That is the honest boundary of this notebook, and of `eye/` itself.